In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import accelerate
import torch

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import accelerate
import torch
import pandas as pd
import numpy as np
os.environ["HUGGINGFACE_TOKEN"] = "INSERT TOKEN"
cache_dir='      '
#model_name = "microsoft/phi-4"
#model_name="google/gemma-2-2b-it"
#model_name="meta-llama/Llama-3.2-3B-Instruct"
#model_name="meta-llama/Llama-3.1-8B-Instruct"
model_name="CohereLabs/aya-expanse-8b"
#model_name="tiiuae/falcon-7b-instruct"
#model_name="mistralai/Mixtral-8x7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=os.getenv("HUGGINGFACE_TOKEN"),cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto", use_auth_token=os.getenv("HUGGINGFACE_TOKEN"),cache_dir=cache_dir)

#model_name = "tiiuae/Falcon3-Mamba-7B-Instruct"

#model = AutoModelForCausalLM.from_pretrained(
#    model_name,
#    torch_dtype=torch.bfloat16,
#    device_map="auto",
# cache_dir=cache_dir
#)
#tokenizer = AutoTokenizer.from_pretrained(model_name)


/home/rag24/.conda/envs/readability/lib/python3.11/site-packages/transformers/models/auto/tokenization_auto.py:823: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/home/rag24/.conda/envs/readability/lib/python3.11/site-packages/transformers/models/auto/auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [12]:
import pandas as pd
import numpy as np
file_path="assetv2.csv"
#file_path=".../vikidia_fr.csv"
df=pd.read_csv(file_path)
df=df.drop(["Unnamed: 0"],axis=1)
#df=df.iloc[:,:5]
df

,ID,first_wins,s,phi_vanilla,phi_avg,phi_entropy,gemma_27b_vanilla,gemma_27b_avg,gemma_27b_entropy,gemma_2b_vanilla,...,conf_gemma_9b_entropy,conf_gemma_9b_verbal_conf,conf_aya_32b_vanilla,conf_aya_32b_avg,conf_aya_32b_entropy,conf_aya_32b_verbal_conf,conf_mixtral_vanilla,conf_mixtral_avg,conf_mixtral_entropy,conf_mixtral_verbal_conf
0,305,1,"Words such as ""Undies"" for underwear and ""movi...",2,1.868187,0.034699,2,1.985693,0.014575,6,...,0.064264,8.987411,3,3.002086,0.012425,7.294206,1,1.029312,1.275783e-02,8.994780
1,1814,2,"Instead, the crew fashioned a trailer. The tra...",2,1.993748,0.020562,3,3.356910,0.056897,6,...,0.072858,7.835491,4,4.201522,0.040977,7.003106,5,4.812475,5.008858e-02,7.880796
2,1787,2,There are statues of Sir Alf Ramsey and Sir Bo...,2,1.948496,0.023660,3,2.695152,0.063067,4,...,0.054981,8.131474,3,2.975553,0.010545,7.651349,1,1.222700,5.112109e-02,8.997528
3,485,2,"The press agreed that Veeck's ""Midget-In-A-Cak...",3,3.355304,0.092215,3,3.464636,0.065896,4,...,0.052778,7.835479,5,4.991029,0.042283,7.002313,5,4.759968,5.154742e-02,7.880797
4,470,1,Since they believed that the grounds were haun...,8,7.484191,0.085900,3,3.385686,0.065740,4,...,0.081570,7.678879,4,4.817015,0.088228,7.002387,5,4.994496,8.971987e-03,7.904650
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
965,554,1,Monteux is a place where religious people live...,2,2.014918,0.028436,4,3.811939,0.062169,4,...,0.064526,8.043818,3,2.997535,0.001789,7.592662,2,2.095633,3.044121e-02,8.060035
966,1280,2,Audiences of the film received extra informati...,7,5.646143,0.145482,4,4.277923,0.098860,4,...,0.079121,7.591922,4,4.263884,0.056693,7.005786,5,5.039745,1.739063e-02,7.499982
967,1286,2,"Unlike the clouds on earth, which are made of ...",2,2.159166,0.043226,3,2.956556,0.052388,3,...,0.074799,7.992742,4,4.217835,0.044593,7.132917,1,1.000431,3.631790e-04,8.997528
968,602,2,About 95 species are currently accepted.,2,1.500779,0.060544,2,1.993984,0.012307,1,...,0.072328,8.983804,2,2.029249,0.010769,7.999712,1,1.000000,2.868446e-12,8.999948


In [13]:
## all models readability scores without confidence
corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    ## Where WIKI texts are
    #s=df.iloc[i,4]
    ## Where Asset texts are 
    s=df.iloc[i,2]
    #score=df.iloc[i,22]
    
    ## Asset prompt
    prompt=f"All of these sentences were rewritten to compare different text simplification methodologies. Rate the readability of the rewritten sentences with a whole number value between 1 (very easy to understand) and 9 (very difficult to understand). Consider factors such as sentence structure, vocabulary or grammar complexity, and overall clarity, as well as your own understanding of readability.  Where on the scale of readability does this text rate: '{s}'. Answer with this format: Answer: [SCORE] Explanation: [EXPLANATION]"    
    
    ## Wikipedia texts prompt
    #prompt=f"These Wikipedia articles are either intended for adult audiences or manually rewritten for children audiences. Rate the readability of each article with a whole number value between 1 (very easy to understand) and 9 (very difficult to understand). Consider factors such as sentence structure, vocabulary or grammar complexity, and overall clarity, as well as your own understanding of readability.  Where on the scale of readability does this text rate: '{s}'. Answer with this format: Answer: [SCORE] Explanation: [EXPLANATION]"    
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=4,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits = scores[-1][0] 
    import torch
    probs = torch.softmax(logits, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-12)).item()
    vocab_size = probs.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs, top6_ids = torch.topk(probs, 6)
    top6_tokens=tokenizer.convert_ids_to_tokens(top6_ids.tolist())
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens[0])
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM',top6_tokens)
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        for (n,p) in zip(top6_tokens,top6_probs.tolist()):
            try:
                avg+=int(n)*p
            except ValueError:
                #print(avg)
                break
        scores_llm_avg.append(avg)
    #real_scores.append(score)
    entropies.append(normalized_entropy)
        #try:
        #    count+=int(tokenizer.decode(outputs[0][-1]))
        #    count1+=1
        #except ValueError:
        #    try:
            #print(tokenizer.decode(outputs[0][-4:]),int(tokenizer.decode(outputs[0][-1])))
        #        count+=int(tokenizer.decode(outputs[0][-2]))
         #       count1+=1
         #   except ValueError:
         #       print('error')
         #       print(tokenizer.decode(outputs[0][-5:]))
   # try:
   #     scores_llm.append(count/count1)
   #     scores.append(score)
   # except ValueError:
   #     print('Big error')
   # except ZeroDivisionError:
   #     print('Zero error')
   #     scores_llm.append('na')
   #     scores.append('na')

#filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
#print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
#corrs.append(np.corrcoef(filtered_scores, filtered_scores_llm)[0, 1])
#print(np.corrcoef(scores, scores_llm)[0, 1])
df['aya_8b_vanilla']=scores_llm_vanilla
df['aya_8b_avg']=scores_llm_avg
df['aya_8b_entropy']=entropies
#print(corrs)
df.to_csv("assetv2.csv")
df.describe()

,ID,first_wins,phi_vanilla,phi_avg,phi_entropy,gemma_27b_vanilla,gemma_27b_avg,gemma_27b_entropy,gemma_2b_vanilla,gemma_2b_avg,...,conf_gemma_9b_entropy,conf_gemma_9b_verbal_conf,conf_aya_32b_vanilla,conf_aya_32b_avg,conf_aya_32b_entropy,conf_aya_32b_verbal_conf,conf_mixtral_vanilla,conf_mixtral_avg,conf_mixtral_entropy,conf_mixtral_verbal_conf
count,970.000000,970.000000,970.000000,970.000000,970.000000,970.000000,970.000000,970.000000,970.000000,970.000000,...,970.000000,970.000000,970.000000,970.000000,970.000000,970.000000,970.000000,970.000000,9.700000e+02,970.000000
mean,1121.729897,1.519588,2.685567,2.706999,0.064530,3.237113,3.269128,0.051597,3.840206,3.926258,...,0.063706,8.053940,3.954639,3.989213,0.037207,7.373811,2.576289,2.593610,2.859217e-02,8.241009
std,625.035508,0.499874,1.562954,1.466178,0.032933,1.262896,1.215501,0.019778,1.762536,1.418764,...,0.022577,0.525787,1.322975,1.290576,0.021830,0.449423,1.667034,1.649354,2.944625e-02,0.572310
min,5.000000,1.000000,1.000000,1.037340,0.013845,1.000000,1.000536,0.000365,1.000000,1.001641,...,0.001005,5.821945,1.000000,1.000006,0.000006,7.000083,1.000000,1.000000,6.447531e-14,6.997008
25%,563.000000,1.000000,2.000000,1.870675,0.039903,2.000000,2.368258,0.039459,2.000000,2.889697,...,0.050485,7.777146,3.000000,3.002417,0.019442,7.012281,1.000000,1.377541,1.357476e-03,7.998837
50%,1145.000000,2.000000,2.000000,2.111866,0.057670,3.000000,3.089587,0.053362,4.000000,4.249992,...,0.062945,7.958390,4.000000,4.015483,0.038727,7.095340,2.000000,2.000002,1.839930e-02,8.119183
75%,1670.000000,2.000000,3.000000,3.001340,0.078374,4.000000,3.875911,0.063915,5.000000,4.997991,...,0.078283,8.056798,5.000000,4.805625,0.054467,7.798184,3.000000,3.107897,5.256940e-02,8.991422
max,2153.000000,2.000000,8.000000,7.920259,0.161499,7.000000,7.471467,0.108476,7.000000,7.079279,...,0.129897,8.999836,7.000000,7.004702,0.098621,8.998498,9.000000,8.777261,1.178203e-01,8.999997


In [11]:
## ## all models readability scores with confidence
corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
verbal_confs=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    
    ##Wikipedia text location
    s=df.iloc[i,4]
    #score=df.iloc[i,22]
    
    ##Asset text location
   # s=df.iloc[i,2]
    
    
    
    ##Asset prompt
    #prompt=f"All of these sentences were rewritten to compare different text simplification methodologies. Rate the readability of the rewritten sentences with a whole number value between 1 (very easy to understand) and 9 (very difficult to understand). Consider factors such as sentence structure, vocabulary or grammar complexity, and overall clarity, as well as your own understanding of readability.  Where on the scale of readability does this text rate: '{s}'. Additionally, state how confident you are that your rating will align with human raters, with a whole number value between 1 and 9. Answer with this format: Answer: [SCORE] Confidence: [Confidence Score] Explanation: [EXPLANATION]"    
    
    
    ##Wikipedia prompt
    prompt=f"These Wikipedia articles are either intended for adult audiences or manually rewritten for children audiences. Rate the readability of each article with a whole number value between 1 (very easy to understand) and 9 (very difficult to understand). Consider factors such as sentence structure, vocabulary or grammar complexity, and overall clarity, as well as your own understanding of readability.  Where on the scale of readability does this text rate: '{s}'. Additionally, state how confident you are that your rating will align with human raters, with a whole number value between 1 and 9. Answer with this format: Answer: [SCORE] Confidence: [Confidence Score] Explanation: [EXPLANATION]"
    messages = [{"role": "user", "content": prompt}] 
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=10,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits_last=scores[-3][0]
    logits_fifth_to_last = scores[-7][0]
    import torch
    probs_last = torch.softmax(logits_last, dim=-1)
    probs_fifth_to_last = torch.softmax(logits_fifth_to_last, dim=-1)
    entropy = -torch.sum(probs_fifth_to_last * torch.log(probs_fifth_to_last + 1e-12)).item()
    vocab_size = probs_fifth_to_last.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
    top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())

    top6_probs_fifth_to_last, top6_ids_fifth_to_last = torch.topk(probs_fifth_to_last, 6)
    top6_tokens_fifth_to_last=tokenizer.convert_ids_to_tokens(top6_ids_fifth_to_last.tolist())

    
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens_fifth_to_last[0])
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM',tokenizer.decode(outputs['sequences'][0][-10:]))
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        for (n,p) in zip(top6_tokens_fifth_to_last,top6_probs_fifth_to_last.tolist()):
            try:
                avg+=int(n)*p
            except ValueError:
                #print(avg)
                break
        scores_llm_avg.append(avg)
        
    avg2=0
    flag2=False
    try:
        vanilla_conf=int(top6_tokens_last[0])
    except ValueError:
        try:
            logits_last=scores[-2][0]
            probs_last = torch.softmax(logits_last, dim=-1)
            top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
            top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
            vanilla_conf=int(top6_tokens_last[0])
        except ValueError:
            try:
                logits_last=scores[-1][0]
                probs_last = torch.softmax(logits_last, dim=-1)
                top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
                top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
                vanilla_conf=int(top6_tokens_last[0])
            except ValueError:
                
                print('ERROR - FIRST CONFIDENCE TOKEN NOT NUM',tokenizer.decode(outputs['sequences'][0][-10:]))
                flag2=True
                verbal_confs.append('na')
    if flag2==False:
        for (n,p) in zip(top6_tokens_last,top6_probs_last.tolist()):
            try:
                avg2+=int(n)*p
            except ValueError:
                #print(avg)
                break
        verbal_confs.append(avg2)
    #real_scores.append(score)
    entropies.append(normalized_entropy)

#filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
#print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
#corrs.append(np.corrcoef(filtered_scores, filtered_scores_llm)[0, 1])
#print(np.corrcoef(scores, scores_llm)[0, 1])
filtered_entropies, filtered_verbal_confs = zip(*[(e,v) for e,v in zip(entropies, verbal_confs) if e != 'na' and v != 'na'])
print(np.corrcoef(filtered_entropies, filtered_verbal_confs)[0, 1])

df['conf_mixtral_vanilla']=scores_llm_vanilla
df['conf_mixtral_avg']=scores_llm_avg
df['conf_mixtral_entropy']=entropies
df['conf_mixtral_verbal_conf']=verbal_confs
#print(corrs)
df.to_csv("vikidia_frv2.csv")

ERROR - FIRST TOKEN NOT NUM Answer: nan
Confidence: 2

-0.12978023594269575
